# Module 05 — Restoring Snapshots

This module covers the full `POST /_snapshot/{repo}/{snapshot}/_restore` API.

## Restore parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `indices` | all in snapshot | Which indices to restore |
| `ignore_unavailable` | `false` | Skip indices not in snapshot |
| `rename_pattern` | — | Regex to match index names |
| `rename_replacement` | — | Replacement string |
| `include_global_state` | `false` | Restore cluster state |
| `feature_states` | `[]` | Feature states to restore |
| `include_aliases` | `true` | Restore aliases |
| `index_settings` | `{}` | Override settings on restore |
| `ignore_index_settings` | `[]` | Settings to strip on restore |
| `partial` | `false` | Restore partial snapshots |
| `wait_for_completion` | `false` | Block until done (query param) |

In [ ]:
from helpers import *

es = get_client()
wait_for_green(es)
register_fs_repo(es)

# Ensure the baseline snapshot exists from Module 00
try:
    es.snapshot.get(repository=FS_REPO_NAME, snapshot='baseline')
    info('Baseline snapshot found.')
except Exception:
    warn('Baseline snapshot not found — run Module 00 first.')

---
## 1. Full Restore

In [ ]:
heading('1. Full restore of all indices from baseline snapshot')

# You cannot restore an index that already exists and is open.
# Strategy: close or delete first, then restore.
es.indices.delete(index='kibana_sample_data_ecommerce', ignore_unavailable=True)
info('Deleted ecommerce index')

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'include_global_state': False,
    },
    wait_for_completion=True,
)

wait_for_restore(es, 'kibana_sample_data_ecommerce')
count = es.count(index='kibana_sample_data_ecommerce')['count']
success(f'Restored ecommerce index: {count} documents')

---
## 2. Selective Restore — Patterns & Wildcards

In [ ]:
heading('2. Selective restore — only flights index')

es.indices.delete(index='kibana_sample_data_flights', ignore_unavailable=True)

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_flights'],  # only this one
        'include_global_state': False,
    },
    wait_for_completion=True,
)
wait_for_restore(es, 'kibana_sample_data_flights')
success(f'Flights index restored: {es.count(index="kibana_sample_data_flights")["count"]} docs')

---
## 3. Rename on Restore

Rename lets you restore a snapshot index to a **different name** — essential when you want
to keep both the live index and the restored copy, or when restoring to a cluster that
already has the index.

In [ ]:
heading('3. Rename on restore')

# Restore ecommerce → ecommerce-restored-copy
es.indices.delete(index='ecommerce-restored-copy', ignore_unavailable=True)

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-restored-copy',
        'include_global_state': False,
        'include_aliases': False,  # don't restore aliases pointing to old name
    },
    wait_for_completion=True,
)

wait_for_restore(es, 'ecommerce-restored-copy')
success(f'Renamed restore succeeded: ecommerce-restored-copy')
success(f'  Original still exists: {es.indices.exists(index="kibana_sample_data_ecommerce")}')
success(f'  Restored copy exists : {es.indices.exists(index="ecommerce-restored-copy")}')

es.indices.delete(index='ecommerce-restored-copy')

---
## 4. Override Settings on Restore

Use `index_settings` to change index configuration during restore —
e.g. reducing replicas for a single-node dev cluster, or changing refresh interval.

In [ ]:
heading('4. Override index settings on restore')

# Show original settings
original = es.indices.get_settings(index='kibana_sample_data_ecommerce')
orig_replicas = original['kibana_sample_data_ecommerce']['settings']['index'].get('number_of_replicas', '?')
orig_refresh = original['kibana_sample_data_ecommerce']['settings']['index'].get('refresh_interval', 'default')
console.print(f'  Original replicas     : {orig_replicas}')
console.print(f'  Original refresh      : {orig_refresh}')

# Restore with overrides
es.indices.delete(index='ecommerce-settings-override', ignore_unavailable=True)

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-settings-override',
        'include_global_state': False,
        'index_settings': {
            'index.number_of_replicas': 0,
            'index.refresh_interval': '30s',
        },
    },
    wait_for_completion=True,
)

wait_for_restore(es, 'ecommerce-settings-override')
restored = es.indices.get_settings(index='ecommerce-settings-override')
new_replicas = restored['ecommerce-settings-override']['settings']['index']['number_of_replicas']
new_refresh = restored['ecommerce-settings-override']['settings']['index'].get('refresh_interval')
success(f'Restored replicas  : {new_replicas}  (was {orig_replicas})')
success(f'Restored refresh   : {new_refresh}  (was {orig_refresh})')

es.indices.delete(index='ecommerce-settings-override')

---
## 5. `ignore_index_settings` — Strip Settings During Restore

Some settings cannot be set on an existing index (e.g. `number_of_shards`).
Others may conflict with the target cluster (e.g. custom analyzers referencing external plugins).
Use `ignore_index_settings` to strip those settings and let Elasticsearch use defaults.

In [ ]:
heading('5. ignore_index_settings')

es.indices.delete(index='ecommerce-stripped-settings', ignore_unavailable=True)

es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-stripped-settings',
        'include_global_state': False,
        # Strip codec and refresh settings — let the target cluster use its defaults
        'ignore_index_settings': ['index.codec', 'index.refresh_interval'],
    },
    wait_for_completion=True,
)

wait_for_restore(es, 'ecommerce-stripped-settings')
success('Restore with stripped settings succeeded')

stripped = es.indices.get_settings(index='ecommerce-stripped-settings')
idx_settings = stripped['ecommerce-stripped-settings']['settings']['index']
console.print(f'  codec in restored index  : {idx_settings.get("codec", "(default — stripped)")} ')
console.print(f'  refresh in restored index: {idx_settings.get("refresh_interval", "(default — stripped)")}')

es.indices.delete(index='ecommerce-stripped-settings')

---
## 6. Restoring Aliases

By default, aliases are restored with their index. You can suppress this with
`include_aliases: false` — useful when you want to restore a renamed copy without
the alias pointing at the old name.

In [ ]:
heading('6. include_aliases')

# Create an alias on the ecommerce index
es.indices.put_alias(index='kibana_sample_data_ecommerce', name='training-alias')
info('Created alias: training-alias → kibana_sample_data_ecommerce')

# Take a snapshot that includes this alias
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-alias-test')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-alias-test',
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': False},
    wait_for_completion=True,
)

# Restore WITH aliases
es.indices.delete(index='ecommerce-with-alias', ignore_unavailable=True)
es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='snap-alias-test',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-with-alias',
        'include_global_state': False,
        'include_aliases': True,
    },
    wait_for_completion=True,
)
wait_for_restore(es, 'ecommerce-with-alias')
aliases_with = es.indices.get_alias(index='ecommerce-with-alias')
console.print(f'  Aliases on restored index (include_aliases=true): {list(aliases_with.get("ecommerce-with-alias", {}).get("aliases", {}).keys())}')

# Restore WITHOUT aliases
es.indices.delete(index='ecommerce-no-alias', ignore_unavailable=True)
es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='snap-alias-test',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-no-alias',
        'include_global_state': False,
        'include_aliases': False,
    },
    wait_for_completion=True,
)
wait_for_restore(es, 'ecommerce-no-alias')
aliases_without = es.indices.get_alias(index='ecommerce-no-alias')
console.print(f'  Aliases on restored index (include_aliases=false): {list(aliases_without.get("ecommerce-no-alias", {}).get("aliases", {}).keys())}')

es.indices.delete(index='ecommerce-with-alias', ignore_unavailable=True)
es.indices.delete(index='ecommerce-no-alias', ignore_unavailable=True)

---
## 7. Restore Specific Feature States

In [ ]:
heading('7. Restore only Kibana feature state (no data indices)')

# Take a snapshot with kibana feature state
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-kibana-feature')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-kibana-feature',
    body={
        'indices': [],
        'include_global_state': False,
        'feature_states': ['kibana'],
    },
    wait_for_completion=True,
)

# Restore only the feature state
es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='snap-kibana-feature',
    body={
        'indices': [],
        'include_global_state': False,
        'feature_states': ['kibana'],  # only Kibana saved objects
    },
    wait_for_completion=True,
)
success('Kibana feature state restored.')

---
## 8. Restoring Global State

In [ ]:
heading('8. Restore global state (templates, pipelines, ILM policies)')

# Take a snapshot with global state
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-global-restore')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-global-restore',
    body={'indices': [], 'include_global_state': True},
    wait_for_completion=True,
)

# Delete the training template to prove it gets restored
try:
    es.indices.delete_index_template(name='training-template')
    warn('Deleted training-template')
except Exception:
    pass

# Restore global state only
es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='snap-global-restore',
    body={
        'indices': [],                # no data
        'include_global_state': True, # YES global state
        'feature_states': ['none'],   # no feature states
    },
    wait_for_completion=True,
)

# Verify template is back
try:
    tpl = es.indices.get_index_template(name='training-template')
    success(f'training-template restored: {tpl["index_templates"][0]["name"]}')
except Exception as e:
    warn(f'Template not found: {e}')

---
## 9. Monitoring Restore Progress

In [ ]:
heading('9. Monitor restore progress')

# Start a restore async
es.indices.delete(index='ecommerce-monitor-test', ignore_unavailable=True)
es.snapshot.restore(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-monitor-test',
        'include_global_state': False,
    },
    wait_for_completion=False,
)

# Poll recovery API
for i in range(20):
    try:
        recovery = es.indices.recovery(index='ecommerce-monitor-test', active_only=True)
        shards = recovery.get('ecommerce-monitor-test', {}).get('shards', [])
        if not shards:
            time.sleep(0.5)
            continue
        for s in shards:
            idx_pct = s.get('index', {}).get('files', {}).get('percent', '?')
            stage = s.get('stage', '?')
            console.print(f'  Shard {s["id"]}: stage={stage}  files={idx_pct}')
            if stage == 'DONE':
                success('Restore complete!')
                break
    except Exception:
        pass
    time.sleep(0.5)

es.indices.delete(index='ecommerce-monitor-test', ignore_unavailable=True)

---
## 10. Restoring to a Different Cluster (Read-Only URL Pattern)

To restore to a different cluster:
1. Make the source cluster's snapshot repository accessible (NFS, object storage, or HTTP)
2. Register it as a `url` or appropriate repository type on the **target** cluster
3. Run `_restore` normally

We simulate this by using the `url-demo` repo created in Module 01.

In [ ]:
heading('10. Cross-cluster restore simulation via URL repo')

# Ensure url-demo repo still exists
try:
    es.snapshot.get_repository(name='url-demo')
except Exception:
    es.snapshot.create_repository(
        name='url-demo',
        body={'type': 'url', 'settings': {'url': f'file://{FS_REPO_PATH}'}},
    )

# List snapshots available in the URL (read-only) repo
url_snaps = es.snapshot.get(repository='url-demo', snapshot='*')
info(f'Snapshots visible via URL repo: {[s["snapshot"] for s in url_snaps["snapshots"]]}')

# Restore from the URL repo (simulating a different cluster reading from it)
es.indices.delete(index='ecommerce-from-url', ignore_unavailable=True)
es.snapshot.restore(
    repository='url-demo',
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-from-url',
        'include_global_state': False,
    },
    wait_for_completion=True,
)
wait_for_restore(es, 'ecommerce-from-url')
success(f'Cross-cluster restore simulation succeeded: ecommerce-from-url')
es.indices.delete(index='ecommerce-from-url')

## Summary

| Scenario | Key parameters |
|----------|---------------|
| Full restore | `indices: ["*"]`, `include_global_state: true`, `feature_states: ["kibana"]` |
| Rename on restore | `rename_pattern` + `rename_replacement` |
| Override settings | `index_settings: {"index.number_of_replicas": 0}` |
| Strip settings | `ignore_index_settings: ["index.codec"]` |
| No aliases | `include_aliases: false` |
| Kibana only | `indices: []`, `feature_states: ["kibana"]` |
| Templates only | `indices: []`, `include_global_state: true`, `feature_states: ["none"]` |
| Monitor progress | `GET /{index}/_recovery` |

**Next:** [06_slm_policies.ipynb](06_slm_policies.ipynb)